In [1]:
import math
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(42)
torch.set_printoptions(precision=3, sci_mode=False)

In [2]:
B, N, Dv = 2, 6, 8
M, Dq, Dlm = 3, 12, 16

visual_tokens = torch.randn(B, N, Dv)
summary = visual_tokens.mean(dim=1)

print("Visual sequence:", tuple(visual_tokens.shape))
print("Pooled summary:", tuple(summary.shape))

reordered_summary = visual_tokens.flip(1).mean(dim=1)

print("Mean unchanged after reordering:",
    torch.allclose(summary, reordered_summary, atol=1e-6)
)

Visual sequence: (2, 6, 8)
Pooled summary: (2, 8)
Mean unchanged after reordering: True


$$\operatorname{Attention}(Q, K, V) = \operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

In [3]:
Q = torch.tensor([
    [1., 0.], [0., 1.]
])

K = torch.tensor([
    [2., 0.], [0., 2.], [1., 1.]
])

V = torch.tensor([
    [10., 0.], [0., 10.], [5., 5.]
])

weights = (Q @ K.T / math.sqrt(K.shape[1])).softmax(dim=-1)
context = weights @ V

print("Attention weights:\n", weights)
print("Collected information:\n", context)
print("Weight sums:", weights.sum(dim=-1))

Attention weights:
 tensor([[0.576, 0.140, 0.284],
        [0.140, 0.576, 0.284]])
Collected information:
 tensor([[7.180, 2.820],
        [2.820, 7.180]])
Weight sums: tensor([1., 1.])


In [4]:
class MiniQueryBridge(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.queries = nn.Parameter(
            torch.randn(1, M, Dq) * 0.02
        )
        self.self_attn = nn.MultiheadAttention(
            Dq, 3, batch_first=True
        )
        self.cross_attn = nn.MultiheadAttention(
            Dq, 3, kdim=Dv, vdim=Dv, batch_first=True
        )
        self.norm1 = nn.LayerNorm(Dq)
        self.norm2 = nn.LayerNorm(Dq)
        self.norm3 = nn.LayerNorm(Dq)
        
        self.ffn = nn.Sequential(
            nn.Linear(Dq, 2 * Dq),
            nn.GELU(),
            nn.Linear(2 * Dq, Dq)
        )
    
    def forward(self, image_features):
        q = self.queries.expand(
            image_features.shape[0], -1, -1
        )
        update, _ = self.self_attn(
            q, q, q, need_weights=False
        )
        q = self.norm1(q + update)
        
        update, attention = self.cross_attn(
            q, image_features, image_features
        )
        q = self.norm2(q + update)
        q = self.norm3(q + self.ffn(q))

        return q, attention

In [5]:
bridge = MiniQueryBridge()
query_features, attention = bridge(visual_tokens)

print("Query features:", tuple(query_features.shape))
print("Attention map:", tuple(attention.shape))

with torch.no_grad():
    longer_features, _ = bridge(torch.randn(B, 10, Dv))

print("With 10 visual tokens:", tuple(longer_features.shape))

Query features: (2, 3, 12)
Attention map: (2, 3, 6)
With 10 visual tokens: (2, 3, 12)


In [7]:
q_to_lm = nn.Linear(Dq, Dlm)
blip_prefix = q_to_lm(query_features)

llava_projector = nn.Sequential(
    nn.Linear(Dv, Dlm),
    nn.GELU(),
    nn.Linear(Dlm, Dlm)
)

llava_prefix = llava_projector(visual_tokens)
text_embedding = nn.Embedding(20, Dlm).requires_grad_(False)

text_ids = torch.tensor([[1, 2, 3, 4], [1, 5, 3, 4]])
text_features = text_embedding(text_ids)

blip_input = torch.cat([blip_prefix, text_features], dim=1)
llava_input = torch.cat([llava_prefix, text_features], dim=1)

print("BLIP-style visual prefix:", tuple(blip_prefix.shape))
print("LLaVA-style visual prefix:", tuple(llava_prefix.shape))
print("BLIP-style combined input:", tuple(blip_input.shape))
print("LLaVA-style combined input:", tuple(llava_input.shape))

BLIP-style visual prefix: (2, 3, 16)
LLaVA-style visual prefix: (2, 6, 16)
BLIP-style combined input: (2, 7, 16)
LLaVA-style combined input: (2, 10, 16)


In [6]:
vision_2d = torch.eye(2)
language_2d = torch.tensor([
    [0., 1.], [1., 0.]
])

aligner = nn.Linear(2, 2, bias=False)
with torch.no_grad():
    aligner.weight.copy_(torch.eye(2))
    
optimizer = torch.optim.SGD(aligner.parameters(), lr=0.5)

before = F.mse_loss(
    aligner(vision_2d), language_2d
).item()

for _ in range(40):
    optimizer.zero_grad()
    loss = F.mse_loss(
        aligner(vision_2d), language_2d
    )
    loss.backward()
    optimizer.step()

after = F.mse_loss(
    aligner(vision_2d), language_2d
).item()

print(f"Before alignment: {before:.3e}")
print(f"After alignment:  {after:.3e}")
print("Aligned vectors:\n", aligner(vision_2d).detach())

Before alignment: 1.000e+00
After alignment:  1.013e-10
Aligned vectors:
 tensor([[    0.000,     1.000],
        [    1.000,     0.000]])


In [8]:
vision_stub = nn.Linear(Dv, Dv)
lm_block = nn.TransformerEncoderLayer(
    Dlm,
    nhead=4,
    dim_feedforward=32,
    dropout=0.0,
    batch_first=True
)

lm_head = nn.Linear(Dlm, 10)
frozen_modules = (vision_stub, lm_block, lm_head)

for module in frozen_modules:
    module.requires_grad_(False)
    module.eval()
    
frozen_params = [
    p for module in frozen_modules
    for p in module.parameters()
]
frozen_before = [
    p.detach().clone() for p in frozen_params
]

In [9]:
projection_before = q_to_lm.weight.detach().clone()
optimizer = torch.optim.Adam(
    list(bridge.parameters()) + list(q_to_lm.parameters()),
    lr=0.01
)

optimizer.zero_grad(set_to_none=True)

image_features = vision_stub(visual_tokens)
query_features, _ = bridge(image_features)
prefix = q_to_lm(query_features)

sequence = torch.cat([prefix, text_features], dim=1)
length = sequence.shape[1]

causal_mask = torch.ones(
    length, length, dtype=torch.bool
).triu(1)

hidden = lm_block(sequence, src_mask=causal_mask)
logits = lm_head(hidden[:, -1, :])

loss = F.cross_entropy(logits, torch.tensor([2, 7]))
loss.backward()

query_gradient = bridge.queries.grad.norm().item()
optimizer.step()

print("Queries received gradient:", query_gradient > 0)
print("Projection updated:",not torch.equal(projection_before, q_to_lm.weight))
print("Frozen parameters received gradients:",any(p.grad is not None for p in frozen_params))
print("Frozen weights unchanged:",all(torch.equal(old, current) for old, current in zip(frozen_before, frozen_params)))

Queries received gradient: True
Projection updated: True
Frozen parameters received gradients: False
Frozen weights unchanged: True
